# Denoising score matching = U-NET model


**Key Takeaways**:
- We use a simple form of a UNet for to predict the noise in the image
- The input is a noisy image, the ouput the noise in the image
- Because the parameters are shared accross time, we need to tell the network in which timestep we are
- The Timestep is encoded by the transformer Sinusoidal Embedding
- We output one single value (mean), because the variance is fixed


In [1]:
import matplotlib.pyplot as plt
from torch import nn
import torch

In [2]:
class Net(nn.Module):
  def __init__(self, in_ch, out_ch, tim_dim, up=False):
    super().__init__()

    self.time_mlp = nn.Linear(tim_dim, out_ch)

    if up:
      self.conv1 = nn.Conv2d(2 * in_ch, out_ch, 3, padding=1)
      self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
    else:
      self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
      self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)

    self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1)
    self.norm1 = nn.BatchNorm2d(out_ch)
    self.norm2 = nn.BatchNorm2d(out_ch)
    self.relu = nn.ReLU()

  def forward(self, x, t):
    x = self.norm1(self.relu(self.conv1(x)))
    t = self.relu(self.time_mlp(t))
    t = t[:, :, None, None]
    x = x + t
    x = self.norm2(self.relu(self.conv2(x)))
    return self.transform(x)


In [3]:
import math
class SinsoidalPosEmbeddings(nn.Module):

  def __init__(self,dim):
    super().__init__()
    self.dim = dim

  def forward(self,t):
    device = t.device
    h = self.dim//2

    embd = math.log(10000) / (h - 1)
    embd = torch.exp(torch.arange(h, device=device) * -embd)

    embd = t[:,None] * embd[None,:]
    embd = torch.cat((embd.sin(), embd.cos()), dim=-1)

    return embd


In [5]:
class SimpleUNet(nn.Module):
    def __init__(self, image_ch=3, time_dim=256):
        super().__init__()

        # 1 Time MLP
        self.time_mlp = nn.Sequential(
            SinsoidalPosEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU()
        )

        # 2) Down blocks
        self.down1 = Net(in_ch=image_ch, out_ch=64,  tim_dim=time_dim, up=False)
        self.down2 = Net(in_ch=64,      out_ch=128, tim_dim=time_dim, up=False)

        # 3 Bottleneck
        self.bot1 = nn.Conv2d(128, 256, 3, padding=1)
        self.bot2 = nn.Conv2d(256, 128, 3, padding=1)

        # 4) Up blocks
        self.up1 = Net(in_ch=128, out_ch=64,  tim_dim=time_dim, up=True)
        self.up2 = Net(in_ch=64,  out_ch=64,  tim_dim=time_dim, up=True)

        # 5) Output layer
        self.out = nn.Conv2d(64, image_ch, 1)

    def forward(self, x, t):
        # time embedding
        t = self.time_mlp(t)

        # downsample
        x1 = self.down1(x, t)
        x2 = self.down2(x1, t)

        # bottleneck
        b = self.bot1(x2)
        b = self.bot2(b)

        # upsample
        u1 = self.up1(b, t)
        u2 = self.up2(u1, t)

        return self.out(u2)


In [8]:
model = SimpleUNet()
print("Num params: ", sum(p.numel() for p in model.parameters()))
model

Num params:  1754115


SimpleUNet(
  (time_mlp): Sequential(
    (0): SinsoidalPosEmbeddings()
    (1): Linear(in_features=256, out_features=256, bias=True)
    (2): ReLU()
  )
  (down1): Net(
    (time_mlp): Linear(in_features=256, out_features=64, bias=True)
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (transform): Conv2d(64, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (norm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (down2): Net(
    (time_mlp): Linear(in_features=256, out_features=128, bias=True)
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (transform): Conv2d(128, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1))
    (norm1): B

## loss -> added noise - pred noise

In [1]:
from forward_diffusion_sample import get_index_from_list, sqrt_one_minus_alphas_cumprod, betas, sqrt_recip_alphas, \
    posterior_variance

@torch.no_grad()
def sample_timestep(x, t, model):
    """
    calls the model to predict the noise residual and returns the denoised image
    """
    betas_t = get_index_from_list(betas, t, x.shape)  # Get the noise value β for the timestep t, and reshape it so it matches the batch shape.
    
    sqrt_one_minus_alphas_cumprod_t = get_index_from_list( 
        sqrt_one_minus_alphas_cumprod, t, x.shape
    )
    
    sqrt_recip_alphas_t = get_index_from_list(sqrt_recip_alphas, t, x.shape)
    
    # Call model (current image - noise prediction)
    model_mean = sqrt_recip_alphas_t * (
        x - betas_t * model(x, t) / sqrt_one_minus_alphas_cumprod_t
    )
    posterior_variance_t = get_index_from_list(posterior_variance, t, x.shape)
    
    if t == 0:
        # As pointed out by Luis Pereira (see YouTube comment)
        # The t's are offset from the t's in the paper
        return model_mean
    else:
        noise = torch.randn_like(x)
        return model_mean + torch.sqrt(posterior_variance_t) * noise 

NameError: name 'torch' is not defined

In [ ]:
from forward_diffusion_sample import IMG_SIZE, T


@torch.no_grad()
def sample_plot_image():
    img_size = IMG_SIZE
    img = torch.randn((1, 3, img_size, img_size), device=device)
    plt.figure(figsize=(15, 15))
    plt.axis('off')
    num_images = 10
    stepsize = int(T / num_images)
    for i in range(0,T)[::-1]:
        t = torch.full((1,), i, device=device, dtype=torch.long)
        img = sample_timestep(img, t)
        # Edit: This is to maintain the natural range of the distribution
        img = torch.clamp(img, -1.0, 1.0)
        if i % stepsize == 0:
            plt.subplot(1, num_images, int(i/stepsize)+1)
            show_tensor_image(img.detach().cpu())
    plt.show()   

## Training

In [2]:
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optim = Adam(model.parameters(), lr=1e-3)
epochs = 100

NameError: name 'torch' is not defined

In [ ]:
for epoch in range(epochs):
    for step, batch in enumerate(tqdm(dataloader)):